In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.autograd import Variable
import numpy as np
import os
from d2l import torch as d2l

In [16]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [17]:
train_iter, vocab = d2l.load_data_time_machine(batch_size = 32, num_steps = 35)

train_iter trả về 1 tuple chứa 2 phần tử phần tử đầu là tensor có shape là batch * num_steps cùng shape với tensor thứ 2. Cả 2 tensor đều là các index trong vocab, các phần tử của tensor 2 là index của kí tự tiếp theo của phần tử tương ứng trong tensor đầu tiên không

chú ý đầu ra của lstm là 1 tuple. phần tử đầu tiên là tensor, phần tử thứ 2 là 1 tuple khác. Đầu vào có dạng batch * num_step * input_dim

In [18]:
class LSTMNoScratch(nn.Module):
    def __init__(self, vocab_size, hidden_dim, batch_first = True):
        super(LSTMNoScratch, self).__init__()
        self.input_dim = vocab_size
        self.hidden_dim = hidden_dim
        self.output_dim = vocab_size
        self.lstm = nn.LSTM(input_size = self.input_dim, hidden_size = hidden_dim, batch_first = batch_first)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X, state = None):
        X = F.one_hot(X, self.input_dim).to(torch.float32)
        out, (hn, cn) = self.lstm(X, state)
        # nếu để batch first là true thì out sẽ có dạng batch * step * hidden
        final_out = self.fc(out)
        return (hn, cn), final_out # trả về state hiện tại và đầu ra (batch*hidden) và (batch*step*vocab_size)

cần output có dạng batch size * step * vocab_size

Perplexity = e^L

In [19]:
next(iter(train_iter))[1].shape

torch.Size([32, 35])

In [20]:
# train

model = LSTMNoScratch(len(vocab), 256).to(device)

epochs = 2000
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    batch_cnt = 0
    for batch_id, (inputs, targets) in enumerate(train_iter):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        output = model(inputs)[1]
        loss = criterion(output.reshape(-1, len(vocab)), targets.reshape(-1))
        pred = output.argmax(dim = 2)
        batch_cnt += 1
        running_loss += loss.item()
        correct += pred.eq(targets.view_as(pred)).sum().item()
        loss.backward()
        optimizer.step()

        if batch_id % 4 == 3:
            training_loss = running_loss / 3
            running_loss = 0
            print(f'training_loss: {training_loss}, perplexity: {torch.exp(torch.tensor(training_loss)).item()}')        
    print(f'accuracy: {correct / (batch_cnt * targets.size(0) * targets.size(1))}')

training_loss: 4.414253393809001, perplexity: 82.6201171875
training_loss: 4.321641604105632, perplexity: 75.31214904785156
accuracy: 0.12890625
training_loss: 4.0914892355601, perplexity: 59.82892990112305
training_loss: 3.9370713233947754, perplexity: 51.26823425292969
accuracy: 0.17165178571428572
training_loss: 3.9099541505177817, perplexity: 49.89665985107422
training_loss: 3.8718978563944497, perplexity: 48.03346252441406
accuracy: 0.16975446428571428
training_loss: 3.87394913037618, perplexity: 48.13208770751953
training_loss: 3.832287152608236, perplexity: 46.168006896972656
accuracy: 0.16975446428571428
training_loss: 3.855326016743978, perplexity: 47.244014739990234
training_loss: 3.8174410661061606, perplexity: 45.48765563964844
accuracy: 0.16975446428571428
training_loss: 3.8378945191701255, perplexity: 46.427616119384766
training_loss: 3.802834431330363, perplexity: 44.82807159423828
accuracy: 0.16975446428571428
training_loss: 3.818543036778768, perplexity: 45.53781127929

In [21]:
class LSTMNoScratch(nn.Module):
    def __init__(self, vocab_size, hidden_dim, batch_first = True):
        super(LSTMNoScratch, self).__init__()
        self.input_dim = vocab_size
        self.hidden_dim = hidden_dim
        self.output_dim = vocab_size
        self.lstm = nn.LSTM(input_size = self.input_dim, hidden_size = hidden_dim, batch_first = batch_first)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X, state = None):
        X = F.one_hot(X, self.input_dim).to(torch.float32)
        out, (hn, cn) = self.lstm(X, state)
        # nếu để batch first là true thì out sẽ có dạng batch * step * hidden
        final_out = self.fc(out)
        return (hn, cn), final_out # trả về state hiện tại và đầu ra (batch*hidden) và (batch*step*vocab_size)

In [23]:

def predict(prefix, num_preds, model, vocab, device='cuda'):
    hn = torch.zeros(size = (1, 1, 256), device = device)
    cn = torch.zeros(size = (1, 1, 256), device = device)
    state = (hn, cn)
    outputs = [vocab.token_to_idx[prefix[0]]]
    def get_input():
        return torch.tensor([outputs[-1]], device = device).to(torch.long).reshape(1,1) # lấy ra output cuối cùng
    for y in prefix[1:]:
        state, fo = model(get_input(), state)
        outputs.append(vocab.token_to_idx[y])
    for _ in range(num_preds):
        state, fo = model(get_input(), state)
        # input là 1*1 output là 1*1*28
        pred = fo.argmax(dim = 2).squeeze().item()
        outputs.append(pred)
    return ''.join([vocab.idx_to_token[i] for i in outputs])
    
predict('time traveller', 234, model, vocab)

'time traveller but now you begin to seethe object of my investigations into the geometry of fourdimensions long ago i had a vague inkling of a machineto travel through time exclaim toa s alw he pathin time for instance here is a portrait of a man a'

# Tóm lại, pipeline sơ lược là:
* training trên dataset traveler.... lấy full hidden state của 1 instance trong batch rồi cho qua fc để lấy ra dự đoán với dim = dim của instance vừa lấy ra
* test gen: init h0 full 0 rồi warm up tính tiếp các h tương ứng với các step trong prefix
* gen tiếp: cho tiếp vào model từ cuối cùng và hidden state cuối cùng cứ làm như thế cho đến khi đạt được num preds